# Simple MLP Training (Direct Dataset Files)

This notebook is basic and direct.

It uses these files as datasets:
- `/data/top_dataset.csv`
- `/data/bottom_dataset.csv`

Classical evaluation settings are kept:
- `RobustScaler`
- `StratifiedKFold(n_splits=10, shuffle=True, random_state=90483257)`
- `cross_val_predict`
- metrics: `accuracy`, `macro_f1`, `balanced_accuracy`

In [ ]:
from pathlib import Path
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

In [ ]:
TOP_FILE = "../data/top_dataset.csv"
BOTTOM_FILE = "../data/bottom_dataset.csv"

## Top File: Load CSV, then Train on ALL Listed Datasets

In [ ]:
top_df = pd.read_csv(TOP_FILE)

print("Top CSV length:", len(top_df))
display(top_df.head(10))

print("\nLoading and preparing all top datasets...")

## Top Dataset: Explicit MLP Code

In [ ]:
# MLP training loop for all top datasets
top_results = []

for idx, row in top_df.iterrows():
    dataset_name = row["dataset"]
    dataset_path = f"../data/{dataset_name}"
    
    try:
        # Load dataset
        data = pd.read_csv(dataset_path, compression="gzip", sep="\t")
        label_col = "class" if "class" in data.columns else "target"
        
        X = data.drop(columns=[label_col]).values.astype(float)
        y = data[label_col].values
        
        # MLP training
        mlp = MLPClassifier(
            hidden_layer_sizes=(100,),
            activation="relu",
            solver="adam",
            max_iter=1000,
            early_stopping=True,
            random_state=324089
        )
        
        pipeline = make_pipeline(RobustScaler(), mlp)
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=90483257)
        pred = cross_val_predict(pipeline, X, y, cv=cv)
        
        # Metrics
        accuracy = accuracy_score(y, pred)
        macro_f1 = f1_score(y, pred, average="macro", zero_division=0)
        balanced_accuracy = balanced_accuracy_score(y, pred)
        
        top_results.append({
            "dataset": dataset_name,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "balanced_accuracy": balanced_accuracy,
        })
        
        print(f"[{idx+1}/{len(top_df)}] {dataset_name}: BA={balanced_accuracy:.4f}")
    except Exception as e:
        print(f"[{idx+1}/{len(top_df)}] {dataset_name}: ERROR - {e}")

top_results_df = pd.DataFrame(top_results)
print(f"\nTop Results ({len(top_results_df)} datasets):")
display(top_results_df)

In [ ]:
# Rank and save top results
rank_metric = "accuracy"

top_ranked_df = top_results_df.sort_values(by=rank_metric, ascending=False).reset_index(drop=True)
top_ranked_df.insert(0, "rank", top_ranked_df.index + 1)

display(top_ranked_df)
top_ranked_df.to_csv("top_mlp_results_ranked.csv", index=False)
print("Top ranked results saved to: top_mlp_results_ranked.csv")

## Bottom File: Load CSV, then Train on ALL Listed Datasets

In [ ]:
bottom_df = pd.read_csv(BOTTOM_FILE)

print("Bottom CSV length:", len(bottom_df))
display(bottom_df.head(10))

print("\nLoading and preparing all bottom datasets...")

## Bottom Dataset: Explicit MLP Code

In [ ]:
# MLP training loop for all bottom datasets
bottom_results = []

for idx, row in bottom_df.iterrows():
    dataset_name = row["dataset"]
    dataset_path = f"../data/{dataset_name}"
    
    try:
        # Load dataset
        data = pd.read_csv(dataset_path, compression="gzip", sep="\t")
        label_col = "class" if "class" in data.columns else "target"
        
        X = data.drop(columns=[label_col]).values.astype(float)
        y = data[label_col].values
        
        # MLP training
        mlp = MLPClassifier(
            hidden_layer_sizes=(100,),
            activation="relu",
            solver="adam",
            max_iter=1000,
            early_stopping=True,
            random_state=324089
        )
        
        pipeline = make_pipeline(RobustScaler(), mlp)
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=90483257)
        pred = cross_val_predict(pipeline, X, y, cv=cv)
        
        # Metrics
        accuracy = accuracy_score(y, pred)
        macro_f1 = f1_score(y, pred, average="macro", zero_division=0)
        balanced_accuracy = balanced_accuracy_score(y, pred)
        
        bottom_results.append({
            "dataset": dataset_name,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "balanced_accuracy": balanced_accuracy,
        })
        
        print(f"[{idx+1}/{len(bottom_df)}] {dataset_name}: BA={balanced_accuracy:.4f}")
    except Exception as e:
        print(f"[{idx+1}/{len(bottom_df)}] {dataset_name}: ERROR - {e}")

bottom_results_df = pd.DataFrame(bottom_results)
print(f"\nBottom Results ({len(bottom_results_df)} datasets):")
display(bottom_results_df)

In [ ]:
# Rank and save bottom results
rank_metric = "accuracy"

bottom_ranked_df = bottom_results_df.sort_values(by=rank_metric, ascending=False).reset_index(drop=True)
bottom_ranked_df.insert(0, "rank", bottom_ranked_df.index + 1)

display(bottom_ranked_df)
bottom_ranked_df.to_csv("bottom_mlp_results_ranked.csv", index=False)
print("Bottom ranked results saved to: bottom_mlp_results_ranked.csv")